# AI Jobs Market 2025–2026: Análise com SQL (DuckDB)

**Autor**: João Victor Neves Alves 
**Dataset**: AI Jobs Market 2025-2026 | Salaries (Kaggle)  
**Objetivo**: Aprofundar os insights da EDA utilizando queries SQL 
analíticas com DuckDB — explorando rankings, deltas de progressão 
salarial e segmentações avançadas.

In [1]:
import duckdb
import pandas as pd

#Carregamento do dataset
df = pd.read_csv('../data/ai_jobs_market_2025_2026.csv')

#Registro do DataFrame como tabela virtual no DuckDB
con = duckdb.connect()
con.register('ai_jobs', df)

print('DuckDB conectado!')
print(f'Tabela "ai_jobs" registrada com {len(df):,} registros')

DuckDB conectado!
Tabela "ai_jobs" registrada com 1,500 registros


## Q1 — Ranking de Salário por Categoria

Posição de cada categoria no ranking salarial, com média, mediana e total de vagas

In [2]:
q1 = con.execute("""
                 SELECT
                    job_category,
                    RANK() OVER (ORDER BY AVG(annual_salary_usd) DESC) AS ranking,
                    ROUND(AVG(annual_salary_usd), 2) AS salario_medio,
                    ROUND(MEDIAN(annual_salary_usd), 2) AS salario_mediano,
                    COUNT(*) AS total_vagas
                 FROM ai_jobs
                 GROUP BY job_category
                 ORDER by ranking
""").df()

print(q1.to_string(index=False))

    job_category  ranking  salario_medio  salario_mediano  total_vagas
    Architecture        1      251576.92         238000.0           52
  AI Engineering        2      207982.34         194500.0          736
  Infrastructure        3      203527.27         196000.0           55
        Security        4      200400.00         194500.0           50
   ML Operations        5      199215.69         189000.0           51
         Product        6      194571.43         185000.0           70
        Research        7      192280.00         183000.0           50
    Data Science        8      181275.59         162000.0          127
Data Engineering        9      176156.86         160000.0           51
        Robotics       10      170851.35         151500.0           74
      Governance       11      152516.39         138500.0          122
        Business       12      134145.16         116000.0           62


**Insights — Q1:**
- **Architecture** lidera o ranking com salário médio de $251k anuais, mas com 
apenas 52 vagas é o nicho mais exclusivo do dataset
- **AI Engineering** é o melhor equilíbrio entre remuneração ($208k) e 
volume (736 vagas — 49% do dataset) — maior empregabilidade do mercado
- A diferença entre média e mediana é expressiva em **Architecture** 
($251k vs $238k) e **AI Engineering** ($208k vs $194k), confirmando 
a presença de salários muito altos puxando a média para cima
- **Business** e **Governance** fecham o ranking — gap de ~$117k em relação 
à Architecture, evidenciando o impacto da especialização técnica
- A window function `RANK()` permite comparar categorias em uma única 
query sem subqueries adicionais, sendo mais eficiente que múltiplos `GROUP BY`

## Q2 — Salário Médio por País e Categoria

In [3]:
q2 = con.execute("""
    WITH base AS (
        SELECT
            country,
            job_category,
            ROUND(AVG(annual_salary_usd), 2) AS salario_medio,
            COUNT(*) AS total_vagas
        FROM ai_jobs
        WHERE country != 'Global'
        GROUP BY country, job_category
        HAVING COUNT(*) >= 5
    )
    SELECT
        country,
        job_category,
        salario_medio,
        total_vagas,
        RANK() OVER (PARTITION BY country
            ORDER BY salario_medio DESC) AS ranking_no_pais
    FROM base
    ORDER BY country, ranking_no_pais
""").df()

print(q2.to_string(index=False))

    country     job_category  salario_medio  total_vagas  ranking_no_pais
  Australia   AI Engineering      212300.00           40                1
  Australia         Robotics      178800.00            5                2
  Australia       Governance      147000.00           10                3
  Australia         Business      110000.00            5                4
     Canada   AI Engineering      190942.22           45                1
     Canada     Data Science      186100.00            6                2
     Canada       Governance      137200.00           10                3
      China     Data Science      147500.00            8                1
      China          Product      140000.00            5                2
      China   AI Engineering      139000.00           47                3
      China         Robotics      125000.00            6                4
      China       Governance      106250.00            8                5
     France          Product      1918

**Insights — Q2:**
- **USA + Architecture** é a combinação mais bem paga do dataset ($285k) — 
seguida de USA + Infrastructure ($243k) e USA + AI Engineering ($241k). 
Os EUA dominam o topo em praticamente todas as categorias
- **AI Engineering aparece como #1 em 7 dos 13 países** — é a categoria 
mais contratada globalmente, independente da região
- **Surpresas regionais:** Singapore lidera com Product ($214k) e UAE com 
ML Operations ($223k) — categorias que não lideram nos EUA mas dominam 
em mercados específicos
- **Switzerland se destaca na Europa** — AI Engineering ($226k) e Security 
($201k) acima da média europeia, refletindo o ecossistema financeiro 
e de tecnologia do país
- **China e India** mostram salários consistentemente abaixo dos demais 
países em todas as categorias — o gap não é apenas nacional mas 
se repete em cada especialização
- O `PARTITION BY country` permite rankear categorias dentro de cada 
país independentemente — mais poderoso que um ranking global único

## Q3 — Top Skills por Categoria

In [6]:
q3 = con.execute("""
    WITH skills_exploded AS (
        SELECT
            job_category,
            annual_salary_usd,
            TRIM(skill) AS skill
        FROM ai_jobs,
        UNNEST(STRING_SPLIT(required_skills, '|')) AS t(skill)
    ),
    skills_ranked AS (
        SELECT
            job_category,
            skill,
            COUNT(*) AS total_vagas,
            ROUND(AVG(annual_salary_usd), 2) AS salario_medio,
            RANK() OVER (PARTITION BY job_category
                 ORDER BY COUNT(*) DESC) AS ranking_skill
        FROM skills_exploded
        GROUP BY job_category, skill
    )
    SELECT *
    FROM skills_ranked
    WHERE ranking_skill <= 5
    ORDER BY job_category, ranking_skill
""").df()

print(q3.to_string(index=False))

    job_category               skill  total_vagas  salario_medio  ranking_skill
  AI Engineering              Python          565      206854.16              1
  AI Engineering             PyTorch          263      208715.59              2
  AI Engineering                 SQL          186      212555.91              3
  AI Engineering            Research          166      210080.72              4
  AI Engineering         Fine-tuning          164      226628.05              5
  AI Engineering               Linux          164      206984.15              5
    Architecture               Cloud           54      258888.89              1
    Architecture          Leadership           48      241520.83              2
    Architecture       System Design           44      253022.73              3
    Architecture                  ML           39      248358.97              4
    Architecture              Python           36      255166.67              5
        Business                 SQL    

**Insights — Q3:**
- **Python é transversal**, aparece no top 5 de 8 das 12 categorias, 
confirmando ser a skill mais universal do mercado de IA independente 
da especialização
- **Cada categoria tem seu DNA técnico:**
  - AI Engineering → Python, PyTorch, Fine-tuning (foco em modelos)
  - Architecture → Cloud, System Design, ML (foco em infraestrutura)
  - ML Operations → Cloud, Kubernetes, Docker, MLflow (foco em deploy)
  - Robotics → C++, Control Systems, Computer Vision (foco em hardware)
  - Governance → EU AI Act, Legal Knowledge, Auditing (foco em regulação)
- **Governance é a única categoria com skill regulatória no top 5** — 
`EU AI Act`, primeira lei abrangente de regulação de Inteligência Artificial do mundo, aprovada pela União Europeia, aparece como requisito, refletindo o impacto crescente 
da regulação europeia de IA no mercado global
- **Fine-tuning em AI Engineering paga o maior salário médio do top 5** 
($227k) — especialização em ajuste fino de modelos é altamente valorizada
- **Data Engineering tem 3 skills empatadas em 3º lugar** — dbt (data build tool), Airflow 
e Python com 36 vagas cada, mostrando que o stack de orquestração 
de dados é igualmente relevante para a categoria
- Duas CTEs encadeadas permitem separar a lógica de explosão de skills 
da lógica de ranking — mais legível e eficiente que uma única query complexa

## Q4 — Progressão Salarial por Nível de Experiência

In [14]:
q4 = con.execute("""

    WITH salario_por_nivel AS (
        SELECT
            experience_level,
            ROUND(AVG(annual_salary_usd), 2) AS salario_medio,
            COUNT(*) AS total_vagas
        FROM ai_jobs
        GROUP BY experience_level
        ORDER BY
            CASE experience_level
                WHEN 'Entry (0-2 yrs)'  THEN 1
                WHEN 'Mid (3-5 yrs)'    THEN 2
                WHEN 'Senior (6-9 yrs)' THEN 3
                WHEN 'Lead (10+ yrs)'   THEN 4
            END
    )
    SELECT
        experience_level,
        salario_medio,
        total_vagas,
        LAG(salario_medio) OVER (
            ORDER BY
                CASE experience_level
                    WHEN 'Entry (0-2 yrs)'  THEN 1
                    WHEN 'Mid (3-5 yrs)'    THEN 2
                    WHEN 'Senior (6-9 yrs)' THEN 3
                    WHEN 'Lead (10+ yrs)'   THEN 4
                END
        ) AS salario_nivel_anterior,
        ROUND((salario_medio - LAG(salario_medio) OVER (
            ORDER BY
                CASE experience_level
                    WHEN 'Entry (0-2 yrs)'  THEN 1
                    WHEN 'Mid (3-5 yrs)'    THEN 2
                    WHEN 'Senior (6-9 yrs)' THEN 3
                    WHEN 'Lead (10+ yrs)'   THEN 4
                END
            )) / LAG(salario_medio) OVER (
                ORDER BY
                    CASE experience_level
                        WHEN 'Entry (0-2 yrs)'  THEN 1
                        WHEN 'Mid (3-5 yrs)'    THEN 2
                        WHEN 'Senior (6-9 yrs)' THEN 3
                        WHEN 'Lead (10+ yrs)'   THEN 4
                    END
            ) * 100
        , 1) AS delta_pct
    FROM salario_por_nivel   
                 
""").df()

print(q4.to_string(index=False))

experience_level  salario_medio  total_vagas  salario_nivel_anterior  delta_pct
 Entry (0-2 yrs)      150038.96          385                     NaN        NaN
   Mid (3-5 yrs)      175983.78          370               150038.96       17.3
Senior (6-9 yrs)      214280.22          364               175983.78       21.8
  Lead (10+ yrs)      240055.12          381               214280.22       12.0


**Insights — Q4:**
- A progressão salarial é consistente em todos os níveis — cada avanço 
de nível representa um aumento real e significativo
- **Entry → Mid: +17.3%** (~$26k) — primeira transição de carreira, 
recompensa a saída do nível júnior
- **Mid → Senior: +21.8%** (~$38k) — o maior salto percentual da carreira 
em IA — confirma o que o boxplot da EDA já indicava visualmente
- **Senior → Lead: +12.0%** (~$26k) — menor delta percentual, mas em 
valor absoluto representa um salto expressivo dado o salário base elevado
- **Entry → Lead acumulado: +60%** — de $150k para $240k ao longo da 
carreira, uma diferença de $90k anuais
- O `LAG()` permite calcular deltas entre linhas de forma elegante — 
o `NULL` em Entry é esperado, pois não há nível anterior para comparar
- O `CASE WHEN` foi necessário para garantir a ordenação hierárquica 
correta — sem ele o SQL ordenaria alfabeticamente, quebrando o cálculo 
do delta

## Q5 — Categorias Acima da Média Geral de Salário

In [16]:
q5 = con.execute("""
                 
    WITH media_geral AS (
        SELECT ROUND(AVG(annual_salary_usd), 2) AS media,
        FROM ai_jobs   
    ),
    media_por_categoria AS (
        SELECT
            job_category,
            ROUND(AVG(annual_salary_usd), 2) AS salario_medio,
            COUNT(*) AS total_vagas
        FROM ai_jobs
        GROUP BY job_category
    )
    SELECT
        m.job_category,
        m.salario_medio,
        m.total_vagas,
        g.media AS media_geral,
        ROUND(m.salario_medio - g.media, 2) AS diferenca_absoluta,
        ROUND((m.salario_medio - g.media) / g.media*100, 1) AS diferenca_pct
    FROM media_por_categoria m
    CROSS JOIN media_geral g
    WHERE m.salario_medio > g.media
    ORDER BY m.salario_medio DESC
                 
""").df()

print(q5.to_string(index=False))

  job_category  salario_medio  total_vagas  media_geral  diferenca_absoluta  diferenca_pct
  Architecture      251576.92           52     194892.0            56684.92           29.1
AI Engineering      207982.34          736     194892.0            13090.34            6.7
Infrastructure      203527.27           55     194892.0             8635.27            4.4
      Security      200400.00           50     194892.0             5508.00            2.8
 ML Operations      199215.69           51     194892.0             4323.69            2.2


**Insights — Q5:**
- Apenas **5 das 12 categorias** superam a média geral do mercado ($194k) — 
o mercado de IA é mais concentrado no topo do que aparenta
- **Architecture se destaca isoladamente** — $56k acima da média geral, 
um prêmio de 29.1% sobre o restante do mercado. A distância para a 
segunda colocada (AI Engineering, +6.7%) é expressiva
- **AI Engineering, Infrastructure, Security e ML Operations** ficam 
próximos da média — diferenças entre 2.2% e 6.7%, indicando que 
essas categorias são bem remuneradas mas não são outliers
- As 7 categorias restantes ficam abaixo da média — puxadas por 
Business ($134k) e Governance ($152k) que reduzem a média geral
- O `CROSS JOIN` com uma CTE de linha única é um padrão elegante para 
comparar cada linha com um valor global — evita subqueries repetidas 
em cada coluna calculada

## Q6 — Impacto da Experiência no Salário por Categoria

In [20]:
q6 = con.execute("""

    WITH salarios AS (
        SELECT
            job_category,
            ROUND(AVG(CASE WHEN experience_level = 'Entry (0-2 yrs)' THEN annual_salary_usd END), 2) AS salario_entry,
            ROUND(AVG(CASE WHEN experience_level = 'Lead (10+ yrs)' THEN annual_salary_usd END), 2) AS salario_lead,
            COUNT(*) AS total_vagas
        FROM ai_jobs
        GROUP BY job_category
    )
    SELECT
        job_category,
        salario_entry,
        salario_lead,
        total_vagas,
        ROUND(salario_lead - salario_entry, 2) AS delta_absoluto,
        ROUND((salario_lead - salario_entry) / salario_entry*100, 1) AS delta_pct,
        RANK() OVER (ORDER BY (salario_lead - salario_entry) DESC) AS ranking_impacto
    FROM salarios
    ORDER BY ranking_impacto

""").df()

print(q6.to_string(index=False))

    job_category  salario_entry  salario_lead  total_vagas  delta_absoluto  delta_pct  ranking_impacto
    Architecture      194100.00     326500.00           52       132400.00       68.2                1
   ML Operations      143800.00     256454.55           51       112654.55       78.3                2
        Security      152750.00     252500.00           50        99750.00       65.3                3
  AI Engineering      155273.22     252191.75          736        96918.53       62.4                4
         Product      158100.00     246181.82           70        88081.82       55.7                5
        Business      101047.62     188250.00           62        87202.38       86.3                6
  Infrastructure      170375.00     251357.14           55        80982.14       47.5                7
      Governance      124090.91     197400.00          122        73309.09       59.1                8
Data Engineering      139235.29     210153.85           51        70918.5

**Insights — Q6:**
- **Architecture tem o maior delta absoluto** ($132k) — um Lead ganha 
$326k vs $194k de um Entry, diferença de 68.2%. A senioridade é 
extremamente valorizada nessa categoria
- **Business tem o maior delta percentual (86.3%)** — apesar de ter 
os menores salários absolutos, a progressão Entry→Lead é a mais 
expressiva proporcionalmente ($101k → $188k)
- **ML Operations combina alto delta absoluto ($112k) e percentual (78.3%)** 
— a categoria com maior crescimento YoY (Q7 da EDA) também é uma 
das que mais recompensa a senioridade
- **Data Science tem o menor delta percentual (36.1%)** — a progressão 
é mais gradual, sugerindo que a senioridade é menos determinante 
que em outras categorias
- **Robotics e Research** também apresentam deltas menores — áreas onde 
conhecimento técnico especializado pode compensar anos de experiência
- A agregação condicional com `CASE WHEN` dentro do `AVG` permite 
pivotar os níveis de experiência em colunas — mais eficiente que 
múltiplos `JOIN` ou subqueries separadas

## Exportação das Querys

In [21]:

q1.to_excel('../outputs/q1_ranking_categorias.xlsx', index=False)
q2.to_excel('../outputs/q2_salario_pais_categoria.xlsx', index=False)
q3.to_excel('../outputs/q3_top_skills_categoria.xlsx', index=False)
q4.to_excel('../outputs/q4_progressao_salarial.xlsx', index=False)
q5.to_excel('../outputs/q5_categorias_acima_media.xlsx', index=False)
q6.to_excel('../outputs/q6_impacto_experiencia.xlsx', index=False)

print('✅ Todos os resultados exportados para outputs/')

✅ Todos os resultados exportados para outputs/


## Conclusões — Análise SQL

As queries analíticas aprofundaram e quantificaram os insights 
identificados na EDA. Abaixo os principais achados:

### Ranking e Remuneração
- **Architecture** lidera com $251k de salário médio — 29.1% acima 
da média geral do mercado ($194k)
- Apenas **5 das 12 categorias** superam a média geral — o mercado 
de IA é mais concentrado no topo do que aparenta
- **AI Engineering** é o melhor equilíbrio entre remuneração ($208k) 
e volume de vagas (736 — 49% do dataset)

### Recorte Regional
- **USA + Architecture** é a combinação mais bem paga ($285k) — 
seguida de USA + Infrastructure ($243k)
- **AI Engineering lidera em 7 dos 13 países** — é a categoria 
mais contratada globalmente
- Surpresas regionais: Singapore lidera com Product ($214k) 
e UAE com ML Operations ($223k)

### Skills por Categoria
- **Python é transversal** — aparece no top 5 de 8 das 12 categorias
- Cada categoria tem seu DNA técnico próprio: ML Operations exige 
Cloud + Kubernetes + Docker, Robotics exige C++ + Control Systems, 
Governance exige EU AI Act + Legal Knowledge
- **Fine-tuning em AI Engineering** paga o maior salário médio 
entre as top skills ($227k)

### Progressão de Carreira
- O maior salto salarial acontece entre **Mid → Senior (+21.8%)** — 
o momento mais crítico da carreira em IA
- **ML Operations** combina alto crescimento YoY com alto delta 
de senioridade (+78.3%) — categoria com maior potencial de valorização
- **Architecture** tem o maior delta absoluto Entry→Lead ($132k) — 
senioridade é extremamente recompensada

### Próximos Passos
- **Fase 3:** Dashboard interativo no Power BI
- **Fase 4:** Apresentação dos insights para stakeholders
- **Fase 5:** Publicação no GitHub e LinkedIn